In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # JobFlow AI — Extração de Skills e Requisitos
# MAGIC
# MAGIC Este notebook extrai skills e requisitos das descrições das vagas.
# MAGIC
# MAGIC Saídas:
# MAGIC
# MAGIC - catálogo inicial de skills
# MAGIC - sentenças classificadas por tipo de requisito
# MAGIC - skills detectadas com evidência textual

# COMMAND ----------

import re
from datetime import datetime, timezone

from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

# COMMAND ----------

CATALOG = "workspace"
SCHEMA = "jobflow_ai"

GOLD_JOBS_TABLE = f"{CATALOG}.{SCHEMA}.gold_job_postings"

SKILLS_CATALOG_TABLE = f"{CATALOG}.{SCHEMA}.skills_catalog"
REQUIREMENT_SENTENCES_TABLE = f"{CATALOG}.{SCHEMA}.gold_job_requirement_sentences"
SKILL_MATCHES_TABLE = f"{CATALOG}.{SCHEMA}.gold_job_skill_matches"

MIN_SENTENCE_CHARS = 25

print("=" * 70)
print("JOBFLOW AI — EXTRAÇÃO DE SKILLS E REQUISITOS")
print("=" * 70)
print(f"Gold jobs table: {GOLD_JOBS_TABLE}")
print(f"Skills catalog table: {SKILLS_CATALOG_TABLE}")
print(f"Requirement sentences table: {REQUIREMENT_SENTENCES_TABLE}")
print(f"Skill matches table: {SKILL_MATCHES_TABLE}")
print(f"Horário UTC: {datetime.now(timezone.utc).isoformat()}")
print("=" * 70)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Carregar vagas Gold

# COMMAND ----------

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

jobs_df = spark.table(GOLD_JOBS_TABLE)

print(f"Registros na Gold: {jobs_df.count()}")

display(
    jobs_df.select(
        "job_id",
        "job_title",
        "company_name",
        "job_location",
        F.length("job_description").alias("description_length"),
    )
    .orderBy(F.col("description_length").desc())
    .limit(10)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Criar catálogo inicial de skills
# MAGIC
# MAGIC Este catálogo é simples e editável. Futuramente ele pode ser expandido
# MAGIC com novas skills, aliases e categorias.

# COMMAND ----------

skills_seed = [
    ("python", "programming_language", ["python", "py"]),
    ("sql", "data", ["sql", "postgresql", "postgres", "mysql", "sql server"]),
    ("spark", "data", ["spark", "apache spark", "pyspark"]),
    ("databricks", "data_platform", ["databricks", "delta lake", "unity catalog"]),
    ("aws", "cloud", ["aws", "amazon web services"]),
    ("azure", "cloud", ["azure", "microsoft azure"]),
    ("gcp", "cloud", ["gcp", "google cloud", "google cloud platform"]),
    ("docker", "devops", ["docker", "container", "containers"]),
    ("kubernetes", "devops", ["kubernetes", "k8s"]),
    ("terraform", "devops", ["terraform", "iac", "infrastructure as code"]),
    ("airflow", "orchestration", ["airflow", "apache airflow"]),
    ("dbt", "data", ["dbt", "data build tool"]),
    ("snowflake", "data_warehouse", ["snowflake"]),
    ("bigquery", "data_warehouse", ["bigquery", "google bigquery"]),
    ("redshift", "data_warehouse", ["redshift", "amazon redshift"]),
    ("java", "programming_language", ["java"]),
    ("scala", "programming_language", ["scala"]),
    ("javascript", "programming_language", ["javascript", "js"]),
    ("typescript", "programming_language", ["typescript", "ts"]),
    ("react", "frontend", ["react", "react.js", "reactjs"]),
    ("node.js", "backend", ["node.js", "nodejs", "node"]),
    ("go", "programming_language", ["golang", "go language"]),
    ("rust", "programming_language", ["rust"]),
    ("machine learning", "ai_ml", ["machine learning", "ml", "scikit-learn", "sklearn"]),
    ("llm", "ai_ml", ["llm", "large language model", "large language models"]),
    ("rag", "ai_ml", ["rag", "retrieval augmented generation"]),
    ("nlp", "ai_ml", ["nlp", "natural language processing"]),
    ("api", "backend", ["api", "apis", "rest api", "restful"]),
    ("microservices", "backend", ["microservices", "microservice"]),
    ("ci/cd", "devops", ["ci/cd", "cicd", "continuous integration"]),
    ("git", "devops", ["git", "github", "gitlab"]),
    ("linux", "systems", ["linux", "unix"]),
    ("etl", "data", ["etl", "elt", "data pipeline", "data pipelines"]),
    ("analytics", "data", ["analytics", "business intelligence", "bi"]),
    ("tableau", "bi", ["tableau"]),
    ("power bi", "bi", ["power bi", "powerbi"]),
]

skills_schema = T.StructType(
    [
        T.StructField("skill_name", T.StringType(), False),
        T.StructField("skill_category", T.StringType(), False),
        T.StructField("aliases", T.ArrayType(T.StringType()), False),
    ]
)

skills_df = spark.createDataFrame(skills_seed, schema=skills_schema)

skills_df = (
    skills_df
    .withColumn("skill_id", F.sha2(F.col("skill_name"), 256))
    .withColumn("created_at", F.current_timestamp())
    .select(
        "skill_id",
        "skill_name",
        "skill_category",
        "aliases",
        "created_at",
    )
)

(
    skills_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SKILLS_CATALOG_TABLE)
)

print(f"OK: catálogo de skills gravado em {SKILLS_CATALOG_TABLE}")

display(skills_df)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Quebrar descrições em sentenças

# COMMAND ----------

sentences_df = (
    jobs_df
    .where(F.col("is_active") == True)
    .where(F.col("job_description").isNotNull())
    .withColumn(
        "description_clean",
        F.trim(F.regexp_replace(F.col("job_description"), "\\s+", " "))
    )
    .withColumn(
        "sentence_array",
        F.split(F.col("description_clean"), r"(?<=[.!?])\s+")
    )
    .select(
        "job_id",
        "source_system",
        "source_job_id",
        "source_job_key",
        "job_title",
        "company_name",
        "job_location",
        "remote_type",
        "tags_text",
        F.posexplode("sentence_array").alias("sentence_index", "sentence_text"),
    )
    .withColumn("sentence_text", F.trim(F.col("sentence_text")))
    .withColumn("sentence_lower", F.lower(F.col("sentence_text")))
    .withColumn("sentence_char_length", F.length(F.col("sentence_text")))
    .where(F.col("sentence_char_length") >= MIN_SENTENCE_CHARS)
)

print(f"Sentenças extraídas: {sentences_df.count()}")

display(sentences_df.limit(20))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Classificar sentenças por tipo

# COMMAND ----------

preferred_pattern = (
    "nice to have|not required|preferred|bonus|plus|desired|optional|"
    "familiarity with|would be nice|would be great|good to have"
)

required_pattern = (
    "required|required qualifications|requirements|must have|need to have|"
    "minimum|at least|experience with|proven experience|strong knowledge|"
    "you have|you bring|qualifications"
)

responsibility_pattern = (
    "you will|responsibilities|responsible for|build|develop|design|"
    "maintain|implement|work on|collaborate|lead|create"
)

benefit_pattern = (
    "benefits|salary|compensation|vacation|health insurance|remote work|"
    "equity|bonus|paid time off"
)

requirement_sentences_df = (
    sentences_df
    .withColumn(
        "requirement_type",
        F.when(F.col("sentence_lower").rlike(preferred_pattern), F.lit("preferred"))
        .when(F.col("sentence_lower").rlike(required_pattern), F.lit("required"))
        .when(F.col("sentence_lower").rlike(responsibility_pattern), F.lit("responsibility"))
        .when(F.col("sentence_lower").rlike(benefit_pattern), F.lit("benefit"))
        .otherwise(F.lit("other"))
    )
    .withColumn(
        "requirement_priority",
        F.when(F.col("requirement_type") == "required", F.lit(1))
        .when(F.col("requirement_type") == "preferred", F.lit(2))
        .when(F.col("requirement_type") == "responsibility", F.lit(3))
        .when(F.col("requirement_type") == "benefit", F.lit(4))
        .otherwise(F.lit(5))
    )
    .withColumn(
        "sentence_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("job_id"),
                F.col("sentence_index").cast("string"),
                F.col("sentence_text"),
            ),
            256,
        )
    )
    .withColumn("processed_at", F.current_timestamp())
    .select(
        "sentence_id",
        "job_id",
        "source_system",
        "source_job_id",
        "source_job_key",
        "job_title",
        "company_name",
        "job_location",
        "remote_type",
        "tags_text",
        "sentence_index",
        "sentence_text",
        "sentence_lower",
        "sentence_char_length",
        "requirement_type",
        "requirement_priority",
        "processed_at",
    )
)

(
    requirement_sentences_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(REQUIREMENT_SENTENCES_TABLE)
)

print(f"OK: sentenças classificadas gravadas em {REQUIREMENT_SENTENCES_TABLE}")

display(
    requirement_sentences_df.groupBy("requirement_type")
    .agg(
        F.count("*").alias("sentences"),
        F.min("requirement_priority").alias("requirement_priority")
    )
    .orderBy("requirement_priority")
)
# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Preparar aliases de skills para matching

# COMMAND ----------

def regex_for_alias(alias: str) -> str:
    alias_lower = alias.lower().strip()
    escaped = re.escape(alias_lower)

    return rf"(^|[^a-z0-9]){escaped}([^a-z0-9]|$)"


alias_rows = []

for skill_name, skill_category, aliases in skills_seed:
    for alias in aliases:
        alias_rows.append(
            {
                "skill_name": skill_name,
                "skill_category": skill_category,
                "alias": alias.lower().strip(),
                "alias_pattern": regex_for_alias(alias),
            }
        )

alias_schema = T.StructType(
    [
        T.StructField("skill_name", T.StringType(), False),
        T.StructField("skill_category", T.StringType(), False),
        T.StructField("alias", T.StringType(), False),
        T.StructField("alias_pattern", T.StringType(), False),
    ]
)

skill_aliases_df = spark.createDataFrame(alias_rows, schema=alias_schema)

display(skill_aliases_df.limit(30))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Detectar skills nas sentenças

# COMMAND ----------

matched_skills_raw_df = (
    requirement_sentences_df.alias("s")
    .crossJoin(skill_aliases_df.alias("a"))
    .where(F.expr("s.sentence_lower RLIKE a.alias_pattern"))
    .select(
        F.col("s.sentence_id"),
        F.col("s.job_id"),
        F.col("s.source_system"),
        F.col("s.source_job_id"),
        F.col("s.source_job_key"),
        F.col("s.job_title"),
        F.col("s.company_name"),
        F.col("s.job_location"),
        F.col("s.remote_type"),
        F.col("s.tags_text"),
        F.col("s.sentence_index"),
        F.col("s.sentence_text").alias("evidence_sentence"),
        F.col("s.requirement_type"),
        F.col("s.requirement_priority"),
        F.col("a.skill_name"),
        F.col("a.skill_category"),
        F.col("a.alias").alias("matched_alias"),
    )
)

print(f"Matches brutos de skills: {matched_skills_raw_df.count()}")

display(matched_skills_raw_df.limit(30))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Deduplicar skill por vaga
# MAGIC
# MAGIC Mantemos a melhor evidência por `job_id + skill_name`.

# COMMAND ----------

dedup_window = (
    Window
    .partitionBy("job_id", "skill_name")
    .orderBy(
        F.col("requirement_priority").asc(),
        F.col("sentence_index").asc(),
        F.length("evidence_sentence").desc(),
    )
)

skill_matches_df = (
    matched_skills_raw_df
    .withColumn("row_number", F.row_number().over(dedup_window))
    .where(F.col("row_number") == 1)
    .drop("row_number")
    .withColumn(
        "skill_match_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("job_id"),
                F.col("skill_name"),
                F.col("matched_alias"),
                F.col("evidence_sentence"),
            ),
            256,
        )
    )
    .withColumn("detected_at", F.current_timestamp())
    .select(
        "skill_match_id",
        "job_id",
        "source_system",
        "source_job_id",
        "source_job_key",
        "job_title",
        "company_name",
        "job_location",
        "remote_type",
        "tags_text",
        "skill_name",
        "skill_category",
        "matched_alias",
        "requirement_type",
        "sentence_index",
        "evidence_sentence",
        "detected_at",
    )
)

(
    skill_matches_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SKILL_MATCHES_TABLE)
)

print(f"OK: matches de skills gravados em {SKILL_MATCHES_TABLE}")

display(skill_matches_df.limit(30))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Validações

# COMMAND ----------

requirements_table_df = spark.table(REQUIREMENT_SENTENCES_TABLE)
skills_table_df = spark.table(SKILL_MATCHES_TABLE)

requirements_summary_df = requirements_table_df.groupBy("requirement_type").agg(
    F.count("*").alias("sentences")
).orderBy("requirement_type")

skills_summary_df = skills_table_df.agg(
    F.count("*").alias("skill_matches"),
    F.countDistinct("job_id").alias("jobs_with_detected_skills"),
    F.countDistinct("skill_name").alias("distinct_skills_detected"),
    F.countDistinct("company_name").alias("distinct_companies"),
)

display(requirements_summary_df)
display(skills_summary_df)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 9. Skills mais frequentes

# COMMAND ----------

display(
    skills_table_df.groupBy("skill_name", "skill_category")
    .agg(
        F.countDistinct("job_id").alias("jobs"),
        F.count("*").alias("matches"),
    )
    .orderBy(F.col("jobs").desc(), F.col("skill_name"))
    .limit(30)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 10. Exemplos de evidências

# COMMAND ----------

display(
    skills_table_df.select(
        "job_title",
        "company_name",
        "skill_name",
        "skill_category",
        "requirement_type",
        "matched_alias",
        F.substring("evidence_sentence", 1, 700).alias("evidence_preview"),
    )
    .orderBy("job_title", "skill_name")
    .limit(30)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 11. Vagas com mais skills detectadas

# COMMAND ----------

display(
    skills_table_df.groupBy(
        "job_id",
        "job_title",
        "company_name",
    )
    .agg(
        F.countDistinct("skill_name").alias("skills_detected"),
        F.concat_ws(", ", F.sort_array(F.collect_set("skill_name"))).alias("skills"),
    )
    .orderBy(F.col("skills_detected").desc(), F.col("job_title"))
    .limit(20)
)

# COMMAND ----------

print()
print("=" * 70)
print("RESULTADO: EXTRAÇÃO DE SKILLS E REQUISITOS CONCLUÍDA")
print("=" * 70)
print(f"skills catalog table: {SKILLS_CATALOG_TABLE}")
print(f"requirement sentences table: {REQUIREMENT_SENTENCES_TABLE}")
print(f"skill matches table: {SKILL_MATCHES_TABLE}")
print(f"sentences: {requirements_table_df.count()}")
print(f"skill matches: {skills_table_df.count()}")
print(f"jobs with detected skills: {skills_table_df.select('job_id').distinct().count()}")
print("=" * 70)